In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Format numbers without scientific notation
pd.set_option('display.float_format', lambda x: '%.2f' % x)

df1 = pd.read_csv('Urb_01.csv', encoding='latin1')
df2 = pd.read_csv('Urb_02.csv', encoding='latin1')
df3 = pd.read_csv('Urb_03.csv', encoding='latin1')
df4 = pd.read_csv('Urb_04.csv', encoding='latin1')
df5 = pd.read_csv('Urb_05.csv', encoding='latin1')
cod = pd.read_csv('codigos.csv')

df = pd.concat([df1, df2, df3, df4, df5], axis=0, ignore_index=True)


In [3]:
# 1. Codificação atribuída ao imóvel [Pesquisador - RU] deve ser inteiros
df['1. Codificação atribuída ao imóvel [Pesquisador - RU]'] = df['1. Codificação atribuída ao imóvel [Pesquisador - RU]'].astype('Int64')
cod['id'] = cod['id'].astype('Int64')

In [4]:
# Merge df and dfPesq with outer join using telefone columns
df = df.merge(cod, left_on='1. Codificação atribuída ao imóvel [Pesquisador - RU]', right_on='id', how='left', suffixes=('', '_cod'))

In [5]:
# Exportar df como csv
df.to_csv('Urb_total.csv', index=False)

In [6]:
# Calculate differences while avoiding scientific notation
df['dif_lat'] = (df['latitude'].abs() - df['Latitude'].abs())
df['dif_lng'] = (df['longitude'].abs() - df['Longitude'].abs())

In [7]:
def haversine_distance(lat1, lon1, lat2, lon2):
  # Convert decimal degrees to radians
  lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
  
  # Haversine formula
  dlat = lat2 - lat1
  dlon = lon2 - lon1
  a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
  c = 2 * np.arcsin(np.sqrt(a))
  r = 6371000  # Earth's radius in meters
  
  return c * r

# Calculate distances
df['distance'] = df.apply(lambda row: haversine_distance(
  row['latitude'], row['longitude'],
  row['Latitude'], row['Longitude']
) if pd.notna(row['Latitude']) and pd.notna(row['Longitude']) else np.nan, axis=1)

# Round to 2 decimal places for better readability
df['distance'] = df['distance'].round(2)

In [ ]:
df.to_csv('Urb_total.csv', index=False)

In [9]:
# Count occurrences of each ID in df and merge with cod dataframe
cod['contagem'] = df['1. Codificação atribuída ao imóvel [Pesquisador - RU]'].value_counts().reindex(cod['id']).fillna(0).astype(int)

# Agrupe contagem de ocorrencias por '1. Codificação atribuída ao imóvel [Pesquisador - RU]' de df
contagem = df['1. Codificação atribuída ao imóvel [Pesquisador - RU]'].value_counts().reset_index()
contagem.columns = ['id', 'contagem']

In [11]:
contagem.to_csv('output_contagem.csv', index=False)

In [9]:
cod.to_csv('output_cod.csv', index=False)